# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Utsabsinha19/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

I will inspect the distributions of impressions, clicks, sessions, average position, CTR, and recent performance change before interpreting the signals. Search and traffic metrics are expected to be uneven, with some pages receiving much more activity than others. The distribution checks help identify heavy tails and prevent a few high-volume pages from being treated as representative of the whole dataset.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

# Create recent impression change
df["impression_change_pct"] = np.where(
    df["impressions_prev_30d"] > 0,
    (df["impressions_last_30d"] - df["impressions_prev_30d"])
    / df["impressions_prev_30d"],
    np.nan
)

fields = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "impression_change_pct"
]

print("Distribution summary:")
display(df[fields].describe().T)

print("\nMedian vs mean:")
for col in fields:
    print(
        f"{col}: mean={df[col].mean():.4f}, "
        f"median={df[col].median():.4f}"
    )

Distribution summary:


,count,mean,std,min,25%,50%,75%,max
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.000000,731.000000,3615.25,517715.0
clicks_90d,30000.0,16.097333,75.076958,0.0,0.000000,1.000000,7.00,4178.0
sessions_90d,30000.0,37.066633,107.069131,1.0,2.000000,7.000000,27.00,4345.0
avg_position,30000.0,16.342380,15.216790,0.0,6.200000,10.800000,22.30,245.0
ctr,30000.0,0.510733,3.279162,0.0,0.000000,0.070000,0.29,100.0
impression_change_pct,26612.0,-0.047857,4.738616,-1.0,-0.626322,-0.334646,0.00,449.0



Median vs mean:
impressions_90d: mean=5200.3663, median=731.0000
clicks_90d: mean=16.0973, median=1.0000
sessions_90d: mean=37.0666, median=7.0000
avg_position: mean=16.3424, median=10.8000
ctr: mean=0.5107, median=0.0700
impression_change_pct: mean=-0.0479, median=-0.3346


## 2. Signal test #1 / #2 / #3 (verdict each)

Signal #1 — Impressions → clicks: I will compare the correlation between 90-day impressions and 90-day clicks and compare low- and high-impression groups.

Signal #2 — Average position → CTR: I will compare CTR across position groups to check whether pages with better observed positions generally have higher CTR.

Signal #3 — Recent impression change: I will compare pages with declining versus non-declining impressions to see whether the decline flag corresponds to a measurable difference in recent performance.

Each signal will receive a verdict based on the observed data rather than on an assumed relationship.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# SIGNAL 1: Impressions vs clicks

corr_1 = df["impressions_90d"].corr(df["clicks_90d"])

df["impression_group"] = pd.qcut(
    df["impressions_90d"],
    q=4,
    duplicates="drop"
)

signal_1 = df.groupby(
    "impression_group",
    observed=True
)["clicks_90d"].agg(["mean", "median", "count"])

print("=== SIGNAL 1: Impressions → Clicks ===")
print("Correlation:", round(corr_1, 4))
display(signal_1)


# SIGNAL 2: Average position vs CTR

df["position_group"] = pd.qcut(
    df["avg_position"],
    q=4,
    duplicates="drop"
)

signal_2 = df.groupby(
    "position_group",
    observed=True
)["ctr"].agg(["mean", "median", "count"])

print("\n=== SIGNAL 2: Average Position → CTR ===")
display(signal_2)


# SIGNAL 3: Recent impression decline

df["declining"] = df["impression_change_pct"] < 0

signal_3 = df.groupby("declining")[
    ["impressions_last_30d", "clicks_last_30d", "ctr"]
].agg(["mean", "median", "count"])

print("\n=== SIGNAL 3: Recent Decline ===")
display(signal_3)

=== SIGNAL 1: Impressions → Clicks ===
Correlation: 0.6963


,mean,median,count
impression_group,,,
"(0.999, 81.0]",0.135812,0.0,7503
"(81.0, 731.0]",0.710628,0.0,7499
"(731.0, 3615.25]",4.337290,2.0,7498
"(3615.25, 517715.0]",59.206800,22.0,7500



=== SIGNAL 2: Average Position → CTR ===


,mean,median,count
position_group,,,
"(-0.001, 6.2]",1.079626,0.10,7543
"(6.2, 10.8]",0.436351,0.12,7534
"(10.8, 22.3]",0.319024,0.10,7462
"(22.3, 245.0]",0.202433,0.00,7461



=== SIGNAL 3: Recent Decline ===


impressions_last_30d               clicks_last_30d                \
                          mean median  count            mean median  count   
declining                                                                    
False              1589.295119   61.0  10284        4.726274    0.0  10284   
True               1345.478342  180.0  19716        5.042149    0.0  19716   

                ctr                
               mean median  count  
declining                          
False      0.866211    0.0  10284  
True       0.325314    0.1  19716

## 3. The flag-linked test

The flag-linked signal is RECENT_DECLINE. The rule assumes that a negative change in impressions is useful as a reason to prioritize a page for review. I will test whether pages with declining impressions show a different distribution of recent clicks, impressions, and CTR from pages without a decline. This does not prove that the page needs content changes; it only tests whether the signal is observable in the data.

In [3]:
print("=== FLAG-LINKED TEST: RECENT_DECLINE ===")

flag_summary = df.groupby("declining")[
    [
        "impressions_last_30d",
        "clicks_last_30d",
        "ctr",
        "avg_position"
    ]
].agg(["mean", "median", "count"])

display(flag_summary)

declining_rate = df["declining"].mean()

print(
    f"\nShare of pages with recent impression decline: "
    f"{declining_rate:.2%}"
)

=== FLAG-LINKED TEST: RECENT_DECLINE ===


impressions_last_30d               clicks_last_30d                \
                          mean median  count            mean median  count   
declining                                                                    
False              1589.295119   61.0  10284        4.726274    0.0  10284   
True               1345.478342  180.0  19716        5.042149    0.0  19716   

                ctr               avg_position                
               mean median  count         mean median  count  
declining                                                     
False      0.866211    0.0  10284    17.300000    9.8  10284  
True       0.325314    0.1  19716    15.842879   11.2  19716


Share of pages with recent impression decline: 65.72%


## 4. What this means in practice

The signal audit suggests that some observable search-performance signals can help prioritize pages for review, but the signals should not be treated as proof that a page needs a content change. The content team should use the ranked results as a directional review queue and combine the signals with human judgment before making changes. Signals that show mixed evidence should receive less weight in later modeling.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Practical takeaway:")
print("Use observed signals to prioritize review, not to automatically change content.")
print("Treat mixed signals cautiously and validate recommendations with human review.")

Practical takeaway:
Use observed signals to prioritize review, not to automatically change content.
Treat mixed signals cautiously and validate recommendations with human review.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.